# [3] Naive Trial with RoBERTa

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import SWUnivDaconDataset

from transformers import pipeline
from torch.utils.data import DataLoader

import pandas as pd
from tqdm.auto import tqdm

import json
import sys

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
pipe = pipeline("text-classification", model="roberta-large-openai-detector")

## Evaluation

In [ ]:
check_only_for = 1
retry_count = 3

In [ ]:
# Validation
corrects, errors, true_human, false_human, results = [], [], [], [], []
progress = tqdm(DataLoader(valid_dataset, batch_size=1, shuffle=True), desc="Validating...")

for idx, data in enumerate(progress):
    label = data[1][0]
    data = data[0][0]

    if check_only_for is not None and label != check_only_for:
        continue  # Skip if the label does not match the specified check

    for trial in range(retry_count):
        assistant_reply = pipe(data)[0]
        predicted = assistant_reply['score']
        if predicted: break
        print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

    result = dict(question=data, label=label, predicted=predicted)
    predicted_label = 1 if predicted['probability'] >= 0.5 else 0
    if predicted_label == label:
        corrects.append(result)
        print(f"INFO: Correct prediction for index {idx}\n\n")
        if label == 0:
            true_human.append(result)
    else:
        result = dict(**result, traceback=assistant_reply)
        errors.append(result)
        print(f"ERROR: Incorrect prediction for index {idx}\n\n")
        if label == 0:
            false_human.append(result)
    results.append(result)
    progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")

In [ ]:
# Test
results = []
for idx, data in enumerate(tqdm(test_dataset, desc="Testing...")):
    data = data[0]

    for trial in range(retry_count):
        assistant_reply = pipe(data)[0]
        predicted = assistant_reply['score']
        if predicted: break
        print(f"WARNING: Invalid prediction for index {idx}, retrying... ({trial + 1}/{retry_count})")

    result = dict(question=data, label=predicted['probability'])
    results.append(result)